In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,classification_report,f1_score,roc_auc_score

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
#Reading Dataset
dataset= pd.read_csv('AI_SocialMedia_Student_Dataset_DP.csv')
dataset

,Age,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Gender_Male,Gender_Non-binary,Education_Level_High School,Education_Level_University
0,19,5,1,7,1,74,98,1,0,0,0
1,16,5,3,7,2,76,89,0,1,1,0
2,25,3,2,6,0,65,76,1,0,0,1
3,23,2,1,6,1,74,87,1,0,0,1
4,20,6,1,5,1,67,88,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...
15995,16,3,4,5,1,76,81,0,0,1,0
15996,14,4,3,5,1,77,96,1,0,1,0
15997,15,0,2,5,1,77,99,0,0,1,0
15998,20,0,5,5,1,80,82,0,0,0,0


In [4]:
#indep=dataset[["Age","Daily_Social_Media_Hours","Mental_Health_Score"]]
#dep=dataset["Gender_Male"]
# Split data into indep and dep
indep = dataset.drop('Mental_Health_Score',axis=1)
dep = dataset['Mental_Health_Score']

In [5]:
indep

,Age,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Physical_Health_Score,Gender_Male,Gender_Non-binary,Education_Level_High School,Education_Level_University
0,19,5,1,7,1,98,1,0,0,0
1,16,5,3,7,2,89,0,1,1,0
2,25,3,2,6,0,76,1,0,0,1
3,23,2,1,6,1,87,1,0,0,1
4,20,6,1,5,1,88,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...
15995,16,3,4,5,1,81,0,0,1,0
15996,14,4,3,5,1,96,1,0,1,0
15997,15,0,2,5,1,99,0,0,1,0
15998,20,0,5,5,1,82,0,0,0,0


In [6]:
dep

0        74
1        76
2        65
3        74
4        67
         ..
15995    76
15996    77
15997    77
15998    80
15999    72
Name: Mental_Health_Score, Length: 16000, dtype: int64

In [13]:
# train_test_split data 
X_train,X_test,y_train,y_test = train_test_split(indep,dep,test_size= 0.1,random_state=0)

#Standard scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Smote for data imbalance
smote = SMOTE(k_neighbors=2,random_state=1)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_smote.value_counts())

ValueError: Expected n_neighbors <= n_samples_fit, but n_neighbors = 3, n_samples_fit = 2, n_samples = 2

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
param_grid={'criterion':['gini', 'entropy', 'log_loss'],'max_features': ['auto','sqrt','log2'],
              'n_estimators':[10,100]}
grid=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1')

# fitting the model for grid search 
grid.fit(X_train_smote, y_train_smote)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,estimator,RandomForestClassifier()
,param_grid,"{'criterion': ['gini', 'entropy', ...], 'max_features': ['auto', 'sqrt', ...], 'n_estimators': [10, 100]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [9]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test_scaled) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
roc_score = roc_auc_score(y_test,grid.predict_proba(X_test_scaled)[:,1])

In [14]:
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("\nThe confusion Matrix:\n",cm)
print("\nThe report:\n",clf_report)
print("\nROC_AUC_Score:",roc_score)

The f1_macro value for best parameter {'criterion': 'gini', 'max_features': 'sqrt', 'n_estimators': 100}: 0.5732904802546019

The confusion Matrix:
 [[949 714]
 [652 885]]

The report:
               precision    recall  f1-score   support

           0       0.59      0.57      0.58      1663
           1       0.55      0.58      0.56      1537

    accuracy                           0.57      3200
   macro avg       0.57      0.57      0.57      3200
weighted avg       0.57      0.57      0.57      3200


ROC_AUC_Score: 0.6290416665525574


In [10]:
table=pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.007454,0.001515,0.000000,0.000000,gini,auto,10,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,13
1,0.005256,0.001251,0.000000,0.000000,gini,auto,100,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,13
2,0.535358,0.014410,0.033695,0.002906,gini,sqrt,10,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.518033,0.527295,0.508603,0.529900,0.605405,0.537847,0.034602,7
3,5.248230,0.078908,0.208001,0.011526,gini,sqrt,100,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.543586,0.562477,0.560031,0.576759,0.635879,0.575746,0.031857,4
4,0.532058,0.004135,0.031458,0.001151,gini,log2,10,"{'criterion': 'gini', 'max_features': 'log2', ...",0.495966,0.511148,0.512559,0.527766,0.588329,0.527153,0.032201,10
5,5.452629,0.115517,0.210088,0.021536,gini,log2,100,"{'criterion': 'gini', 'max_features': 'log2', ...",0.549847,0.572816,0.591377,0.563326,0.634512,0.582375,0.029366,1
6,0.005770,0.000907,0.000000,0.000000,entropy,auto,10,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,13
7,0.005190,0.000491,0.000000,0.000000,entropy,auto,100,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,13
8,0.607803,0.018495,0.033389,0.003600,entropy,sqrt,10,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.504160,0.494118,0.519934,0.509153,0.590536,0.523580,0.034495,12
9,6.379458,0.185091,0.218117,0.013933,entropy,sqrt,100,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.551163,0.553239,0.564868,0.577999,0.629118,0.575277,0.028572,5


In [11]:
age = int(input("Enter age:"))
daily_socialmedia_hours = int(input("Enter amount spent on socialmedia:"))
daily_ai_tool_usage = int(input("Daily_AI_Tool_Usage_Hours:"))
daily_sleep_hours = int(input("Sleep_Hours:"))
daily_physical_activity_hours = int(input("Physical_Activity_Hours:"))
mental_health_score = int(input("Mental_Health_Score"))
physical_health_score = int(input("Physical_Health_Score"))
transgender = int(input("Transgender (1-Yes , 0-No)" ))
edu_high_school = int(input("Education_Level_High School"))
edu_university = int(input("Education_Level_University"))

Enter age: 45
Enter amount spent on socialmedia: 5
Daily_AI_Tool_Usage_Hours: 0
Sleep_Hours: 6
Physical_Activity_Hours: 1
Mental_Health_Score 56
Physical_Health_Score 56
Transgender (1-Yes , 0-No) 1
Education_Level_High School 1
Education_Level_University 0


In [12]:
user_data=[[age,daily_socialmedia_hours,daily_ai_tool_usage,daily_sleep_hours,daily_physical_activity_hours,mental_health_score,physical_health_score,transgender,edu_high_school,edu_university]]
user_data_scaled = scaler.transform(user_data)
future_prediction = grid.predict(user_data_scaled)
print("Mental_Health_Score =", future_prediction[0])

Male or Femal Prediction = 0
